# Узел А — подготовка CSV и HTTP-запрос

Пошаговая шпаргалка для ручной работы на хосте А. Команды запускайте в терминале: ячейки не запускают Docker и не передают файл автоматически. Сначала выполните общую настройку из корневого README.


## 1. Запустить контейнер

Из корня проекта на хосте А выполните:

```bash
cd node_a
cp .env.example .env
```

Команда копирования нужна только при первоначальной настройке. Не выполняйте её поверх уже настроенного `.env`. Проверьте `TARGET_COLUMN=species`, затем соберите и запустите idle-контейнер:

```bash
docker compose up --build -d
docker ps --filter name=data-processor
```


## 2. Подготовить исходный CSV

Для встроенного Iris выполните отдельную команду из любого каталога хоста:

```bash
docker exec data-processor python /app/download_dataset.py
```

Скрипт экспортирует встроенный Iris из `sklearn` в `/home/DATA/raw_data.csv`; сетевого скачивания нет. Существующий файл он не заменит. Если заменяете его намеренно:

```bash
docker exec data-processor python /app/download_dataset.py --overwrite
```

Вместо генерации положите выданный файл как `node_a/data/raw_data.csv` на хосте А или передайте его в контейнер через `docker cp`. Для своего размеченного CSV задайте одинаковый `TARGET_COLUMN` в `.env` обоих узлов; поддерживаются числовые и категориальные табличные признаки.


## 3. Очистить CSV

Запустите очистку отдельно:

```bash
docker exec data-processor python /app/clean_data.py
```

Скрипт читает `/home/DATA/raw_data.csv` и записывает `/home/DATA/clean_data.csv`. Контейнерный `/home/DATA` соответствует `node_a/data` на хосте. Очистка не выполняет SSH, SCP, curl или генерацию исходного набора.


## 4. Передать CSV на хост Б

На хосте А из корня проекта проверьте файл и отправьте его вручную. Подставьте IP, имя пользователя и абсолютный путь каталога `node_b/data` на хосте Б:

```bash
ls -l node_a/data/clean_data.csv
scp -i ~/.ssh/lab2_ed25519 node_a/data/clean_data.csv USER_B@IP_B:/ABS/PATH/project/node_b/data/clean_data.csv
```

Если терминал уже находится в каталоге `node_a`, укажите источник `data/clean_data.csv`. Приватный ключ остаётся на хосте А и не монтируется в контейнер.


## 5. Запросить готовую модель

Убедитесь, что на хосте Б модель уже обучена и Flask запущен в foreground. Затем с хоста А выполните:

```bash
docker exec data-processor curl http://IP_B:5000/health
docker exec data-processor curl http://IP_B:5000/predict
```

`/health` возвращает `status: "ok"`, `model_ready` и `test_data_ready`. Успешный `/predict` возвращает `status: "success"`, `train_rows: 120`, `test_rows: 30`, accuracy и 30 predictions с полями `row_index` и `prediction`. `row_index` — индекс строки очищенного CSV, начиная с 0. До обучения `/predict` отвечает `503`; запрос не обучает модель и не делит данные повторно. `ping IP_B` проверяет ICMP, а не доступность TCP-порта 5000. В отчёт вручную добавьте требуемые преподавателем скриншоты.
